In [1]:
%%sql
SELECT delivery_status, transaction_type, count(*) n, round(sum(final_amount)) revenue
FROM gold.fact_eod_sale_product GROUP BY 1,2 ORDER BY 1,2;

StatementMeta(, 007c0f4c-0af8-48e9-a3f7-19124f593d04, 4, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 4 fields>

In [2]:
%%sql
SELECT report_date, count(*) FROM gold.fact_eod_sale_product GROUP BY 1;

StatementMeta(, 007c0f4c-0af8-48e9-a3f7-19124f593d04, 5, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [3]:
%%sql
SELECT hour(transaction_time) hh, count(*) n FROM gold.fact_eod_sale_product GROUP BY 1 ORDER BY 1;

StatementMeta(, 007c0f4c-0af8-48e9-a3f7-19124f593d04, 6, Finished, Available, Finished, False)

<Spark SQL result set with 22 rows and 2 fields>

In [4]:
%%sql
SELECT sum(CASE WHEN transaction_id IS NULL THEN 1 ELSE 0 END) null_txn,
       sum(CASE WHEN product_id     IS NULL THEN 1 ELSE 0 END) null_prod,
       sum(CASE WHEN report_date    IS NULL THEN 1 ELSE 0 END) null_date
FROM gold.fact_eod_sale_product;

StatementMeta(, 007c0f4c-0af8-48e9-a3f7-19124f593d04, 7, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [5]:
%%sql 
SELECT typeof(saleNormalItems), saleNormalItems FROM bronze.sale_bill LIMIT 1

StatementMeta(, 007c0f4c-0af8-48e9-a3f7-19124f593d04, 8, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [6]:
row = spark.table("bronze.sale_bill").select("saleNormalItems").first()[0]
print(type(row)); print(row[:800])

StatementMeta(, 007c0f4c-0af8-48e9-a3f7-19124f593d04, 10, Finished, Available, Finished, False)

<class 'str'>
[{"productCode":"P18","productName":"Product P18","uom":"EA","uomSize":1,"quantity":1,"totalAmount":12000,"totalAmountWithoutTax":10909.09,"totalAmountByPromotion":10800,"totalAmountWithoutTaxByPromotion":9818.18,"winPromotion":true,"productGroupId":12,"productGroup":"Ready Meals","productSubCategoryId":122,"productSubCategory":"Ready Meals Sub","productCategoryId":1000,"productCategory":"Food","productUomId":4,"retailSellingPrice":12000,"retailBusinessType":"Merchandise","outputVatCode":"V10"},{"productCode":"P02","productName":"Product P02","uom":"EA","uomSize":1,"quantity":4,"totalAmount":60000,"totalAmountWithoutTax":54545.45,"totalAmountByPromotion":54000,"totalAmountWithoutTaxByPromotion":49090.9,"winPromotion":false,"productGroupId":12,"productGroup":"Ready Meals","productSubCategor


In [7]:
%%sql
SELECT count(*) total, count(saleNormalItems) non_null
FROM bronze.sale_bill;

StatementMeta(, 007c0f4c-0af8-48e9-a3f7-19124f593d04, 11, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [4]:
%%sql
CREATE OR REPLACE VIEW gold.vw_eod_sales AS
SELECT
    f.report_date,
    year(f.report_date)                                   AS year,
    date_format(f.report_date, 'yyyy-MM')                 AS month_key,
    date_trunc('WEEK', f.report_date)                     AS week_start,   -- Monday
    date_format(f.report_date, 'EEE')                     AS day_name,
    CASE WHEN dayofweek(f.report_date) IN (1, 7) THEN 1 ELSE 0 END AS is_weekend,  -- 1=Sun,7=Sat
    hour(f.transaction_time)                              AS txn_hour,
    CASE
        WHEN hour(f.transaction_time) BETWEEN 6  AND 10 THEN '1 Morning (06-10)'
        WHEN hour(f.transaction_time) BETWEEN 11 AND 13 THEN '2 Lunch (11-13)'
        WHEN hour(f.transaction_time) BETWEEN 14 AND 16 THEN '3 Afternoon (14-16)'
        WHEN hour(f.transaction_time) BETWEEN 17 AND 21 THEN '4 Evening (17-21)'
        ELSE '5 Night (22-05)'
    END                                                   AS daypart,
    CASE
        WHEN nullif(f.transaction_7now_source, '') IS NOT NULL       THEN '7NOW'
        WHEN upper(coalesce(f.channel_name, '')) LIKE '%SAVYU%'      THEN 'Savyu'
        WHEN upper(coalesce(f.channel_name, '')) LIKE '%7NOW%'       THEN '7NOW'
        WHEN nullif(f.channel_name, '') IS NOT NULL                  THEN f.channel_name
        ELSE 'Store'
    END                                                   AS channel,
    f.store_id,
    f.store_name,
    f.store_type,
    f.transaction_id,
    f.product_id,
    f.quantity,
    f.final_amount_no_tax                                 AS revenue_net,
    f.final_amount                                        AS revenue_gross,
    f.customer_code,
    CASE WHEN nullif(f.customer_code, '') IS NOT NULL THEN 1 ELSE 0 END AS is_identified_customer
FROM gold.fact_eod_sale_product f
WHERE f.delivery_status = 'completed'
  AND f.transaction_type = 'Sale Transaction'
  AND f.store_id NOT IN ('1000', '1004')
  AND f.store_id NOT LIKE '19%';

StatementMeta(, 262cbb16-5980-463d-a598-b8075a2cc904, 8, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
%%sql
CREATE OR REPLACE VIEW gold.vw_eod_sales_daily AS
SELECT
    report_date,
    year,
    month_key,
    week_start,
    is_weekend,
    channel,
    count(DISTINCT transaction_id)                        AS baskets,
    count(DISTINCT store_id)                              AS stores,
    sum(quantity)                                         AS units,
    round(sum(revenue_net), 0)                            AS revenue_net,
    round(sum(revenue_gross), 0)                          AS revenue_gross,
    round(sum(revenue_net) / nullif(count(DISTINCT transaction_id), 0), 0) AS basket_size
FROM gold.vw_eod_sales
GROUP BY report_date, year, month_key, week_start, is_weekend, channel;

StatementMeta(, 262cbb16-5980-463d-a598-b8075a2cc904, 9, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [6]:
%%sql
CREATE OR REPLACE TABLE gold.bi_eod_sales       USING DELTA AS SELECT * FROM gold.vw_eod_sales;
CREATE OR REPLACE TABLE gold.bi_eod_sales_daily USING DELTA AS SELECT * FROM gold.vw_eod_sales_daily;

StatementMeta(, 262cbb16-5980-463d-a598-b8075a2cc904, 11, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
spark.sql("SHOW TABLES IN gold").show(truncate=False)
spark.sql("DESCRIBE DETAIL gold.bi_eod_sales").select("format","location").show(truncate=False)

StatementMeta(, 262cbb16-5980-463d-a598-b8075a2cc904, 12, Finished, Available, Finished, False)

+--------------------------------------+---------------------+-----------+
|namespace                             |tableName            |isTemporary|
+--------------------------------------+---------------------+-----------+
|RetailSales_Analysis.lh_eod_sales.gold|bi_eod_sales         |false      |
|RetailSales_Analysis.lh_eod_sales.gold|bi_eod_sales_daily   |false      |
|RetailSales_Analysis.lh_eod_sales.gold|fact_eod_sale_product|false      |
|RetailSales_Analysis.lh_eod_sales.gold|vw_eod_sales         |false      |
|RetailSales_Analysis.lh_eod_sales.gold|vw_eod_sales_daily   |false      |
+--------------------------------------+---------------------+-----------+

+------+-------------------------------------------------------------------------------------------------------------------------------------------+
|format|location                                                                                                                                   |
+------+------------------

In [8]:
%%sql
CREATE OR REPLACE TABLE gold.dim_date AS
SELECT
    d                                            AS date,
    year(d)                                      AS year,
    date_format(d, 'yyyy-MM')                    AS month_key,
    date_format(d, 'MMM yyyy')                   AS month,
    date_trunc('WEEK', d)                        AS week_start,        -- Monday
    date_format(d, 'EEE')                        AS day_name,
    ((dayofweek(d) + 5) % 7) + 1                 AS weekday_no,        -- 1=Mon..7=Sun (để sort day_name)
    CASE WHEN dayofweek(d) IN (1,7) THEN true ELSE false END AS is_weekend
FROM (
    SELECT explode(sequence(
        (SELECT min(report_date) FROM gold.bi_eod_sales),
        (SELECT max(report_date) FROM gold.bi_eod_sales),
        interval 1 day)) AS d
);

StatementMeta(, 262cbb16-5980-463d-a598-b8075a2cc904, 13, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>